## Programa para Adquirir Dados

In [ ]:
import os
import time
import sys
import clr
import numpy as np
import matplotlib.pyplot as plt
import time 
from pylablib.devices import Thorlabs
from structured_optics import *
from structured_optics.hologram import slm_hologram
from structured_optics.slmdisplay import My_SLM
from tqdm import tqdm
from camera import *
import h5py

# Add References to .NET libraries

clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\Thorlabs.MotionControl.DeviceManagerCLI.dll")
clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\Thorlabs.MotionControl.GenericMotorCLI.dll")
clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\ThorLabs.MotionControl.KCube.StepperMotorCLI.dll")
from Thorlabs.MotionControl.DeviceManagerCLI import *
from Thorlabs.MotionControl.GenericMotorCLI import *
from Thorlabs.MotionControl.KCube.StepperMotorCLI import *
from System import Decimal  # necessary for real world units


## Funções

In [ ]:
def calculate_fix_std_LG(C, l, p, z):
    return np.sqrt(2*C**2/(2*p + np.abs(l) +1) + 2*np.sqrt(np.abs(C**4/(2*p + np.abs(l) +1)**2 - z**2/beam.k()**2)))/mu

def calculate_fix_std_HG(C, n, z):
    return np.sqrt(2*C**2/(2*n +1) + 2*np.sqrt(np.abs(C**4/(2*n +1)**2 - z**2/beam.k()**2)))/mu

## Parâmetros

In [ ]:
pixel_size = 8e-6  #tamanho do pixel do slm 
lamb = 633e-9   #comprimento de onda
waist = 0.5e-3  #cintura do feixe que sai do slm
pixel_cam = 5.6e-6

mu = 1 #parâmetro de magnificação devido ao sistema 4F após o SLM
z = 20e-2 #Distância entre o waist pós 4F e a amostra, utilizado para calcular fix std
C = 0.5e-3  #std a ser fixado

resolution = 100  #Resolução dos motores de passo
step_size = 100e-9 #tamanho do passo dos motores de passo

EXPO = 1   #exposição camera em segundos
BIT = 8     #bitdepth da camera

NUM_REP = 5  #numero de repetições da medida
AMOSTRA = "Ferro"   #tipo de amostra
ANGULO_AMOSTRA = 90 #angulo das ranhuras. 90 representa vertical, 45 diagonal e 0 horizontal.



## Primeiro alinhamento e check de equipamentos

In [ ]:
#Inicializa e set home dos motores de passo

stage_z = Thorlabs.kinesis.KinesisMotor('26005619',scale='ZST',  default_channel=2, is_rack_system=False)
print(f" stage z  -- {stage_z.get_position(scale= 'ZST')}")
stages = [stage_z]

for _ in stages:
    print(_.is_homed())
    if _.is_homed() == False:
        _.home()
for _ in stages:
    _.wait_for_home()

stage_z.move_to(0.004) # in meter
stage_z.wait_move()

In [ ]:
#inicializa camera, slm e lista de modos a serem medidos (até ordem 5)

slm = My_SLM(1)
width = slm.width
height = slm.height

cam = init_camera(bit_depth=BIT, exposure_s = EXPO)


hg_modes = [[0,0], [0,1], [1,0], [2,0], [1,1], [0,2], [3,0], 
            [2,1], [1,2], [0,3], [4,0], [3,1], [2,2], [1,3], 
            [0,4], [5,0], [4,1], [3,2], [2,3], [1,4], [0,5]]
lg_modes = [[0,0], [0,1], [0,-1], [0,2], [1,0], [0,-2], [0,3], 
            [1,1], [1,-1], [0,-3], [0,4], [1, 2], [1,-2], [2, 0], 
            [0,-4], [0,5], [1,3], [1, -3], [2, 1], [2,-1], [0,-5]]

In [ ]:
#inicializa objeto beam que contém os parâmetros do feixe gerado

width = slm.width
height = slm.height

beam = Beam(nix = pixel_size*width/2, Dx = width,
            niy = pixel_size*height/2, Dy = height,
            waist = waist, lamb = lamb)

In [ ]:
#Define qual o campo do objeto beam, gera um holograma a partir desse campo e atualiza a máscara do slm com esse holograma

beam.lg(2,2)
#beam.triangle()
holo = slm_hologram(beam, 3,4)
slm.update_image(holo)

## Dados LG waist normal

In [ ]:

with h5py.File(f"data_LG_amostra{AMOSTRA}_orientacao{ANGULO_AMOSTRA}", "w") as file:
    for mode in lg_modes:
        data_lg = np.empty((NUM_REP, resolution,480, 640),dtype="uint8")
        l = mode[1]
        p = mode[0]
        mode_group = file.create_group(f"LG{p}{l}")

        beam.lg(l,p)
        holo = slm_hologram(beam, 3, 4)
        slm.update_image(holo)
        time.sleep(0.2)
        for rep in range(NUM_REP):
            stage_z.move_to(0.003) # in meter
            stage_z.wait_move()
            stage_z.move_to(0.004) # in meter
            stage_z.wait_move()
            for i in range(resolution):
                stage_z.move_by(step_size)
                stage_z.wait_move()
                time.sleep(0.1)
                data_lg[rep, i] = take_picture(cam)

        images_ds = mode_group.create_dataset("images", (NUM_REP, resolution,480, 640), dtype='uint8')
        images_ds[:] = data_lg[:]

    file.attrs['expo'] = EXPO
    file.attrs['bit_depth'] = BIT
    file.attrs['step_size'] = str(int(step_size*1e9))+'nm'
    




